# 미션: 새로운 모델을 만들고 100,000건을 가능한 빠르게 데이터베이스에 적재하세요.

목표는 100,000건을 넣는 것 자체가 아니라 적재 성능을 개선하는 것입니다.

데이터는 일부만 보여주거나 개수만 보여주세요.

---

### 이슈 1
외부에서 benchmark를 수행할 방법을 아무리 찾아도 모르겠어서 GPT에 질문하였고 Django 초기화가 필요하다는 것을 알게됨

따라서 해당 구문을 이해하기 위해 아래 문서를 참고함.
- [python path](https://docs.python.org/3/using/cmdline.html#envvar-PYTHONPATH)

- [Initialization process](https://docs.djangoproject.com/en/6.0/ref/applications/#initialization-process)
- [calling-django-setup-is-required-for-standalone-django-usage](https://docs.djangoproject.com/en/4.2/topics/settings/#calling-django-setup-is-required-for-standalone-django-usage)
- [django-admin and manage.py](https://docs.djangoproject.com/en/6.1/ref/django-admin/)

결국 이해 못했고 아래 코드에 대해 다시 분석 시작...



```python
# 문제의 Django 초기화

import os
import sys
from pathlib import Path

MONOREPO = Path("/Users/ahh/bootcamp-playground/chapter2/monorepo")

# 이해가 되지 않았던 문법 시작
if str(MONOREPO) not in sys.path:

    # 1. sys.path에는 insert라는 메서드가 확인되지 않는데 어떻게 사용했는지?
    # 2. sys.path가 리스트로 관리되어서 리스트 메서드인 insert 구문을 사용할 수 있다고 하는데 그건 또 어디서 확인하는지?
    #   - [공식 doc](https://docs.python.org/3/library/sys.html#sys.path)
    #   - `A list of strings that specifies the search path for modules.`
    #   - `Only strings should be added to sys.path;`

    #   - 결국 해석하면 sys.path에서 관리하는 import 경로 List의 0번 index에 str(Path)를 추가하라는거네.
    #   - 무조건 str로 추가하라고 되어있어서 멀쩡한 Path를 str로 변환한거고...
    sys.path.insert(0, str(MONOREPO))

# 3. os.environ에는 setdefault라는 메서드를 아무리 찾아도 없는데 어떻게 사용되었는지?
#   - environ은 또 뭐냐... env가 환경인건 알겠는데...
#   - [os.environ](https://docs.python.org/3/library/os.html#os.environ)
#   - `A mapping object where keys and values are strings that represent the process environment.`
#   - 그러니까 프로세스 환경관리가 mapping object로 이루어진다는 거네. 키랑 벨류랑 매칭해서
#   - [setdefault](https://docs.python.org/ko/3/library/stdtypes.html#dict.setdefault
#   - `key 가 딕셔너리에 있으면 해당 값을 돌려줍니다. 그렇지 않으면, default 값을 갖는 key 를 삽입한 후 default 를 돌려줍니다. default 의 기본값은 None 입니다.`
#   - 그러니까 JANGO_SETTINGS_MODULE가 이미 들어있으면 그 value를 return하고 아니면 config.settings를 삽입하겠다고...?
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")

"""
그러니까 여기까지 종합하면 pathlib으로 경로를 작성해주고 이걸 sys.path에 0번 인덱스에 문자열로 바꿔서 추가한다.
위의 이유는 경로를 추적하고 저기에 있는 파일들을 임포트 하려는 거고...

아래는 os. 프로세스 환경변수에서 저 DJANGO_SETTINGS_MODULE이 있으면 그냥 넘어가고 없으면 오른쪽 값하고 같이 k-v 구조로 저장하겠다고...

정리하면 `sys.path.insert(0, str(MONOREPO))`는 python에다가 임포트할 파일이 위치한 폴더를 알려준거고
`os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")`는 Django한테 설정파일 위치를 알려준거고...
"""


# 장고 수동 초기화.
# [Django Settings](https://docs.djangoproject.com/en/6.0/topics/settings/?utm_source=chatgpt.com#calling-django-setup-is-required-for-standalone-django-usage)
#   - `If you’re using components of Django “standalone” – for example, writing a Python script which loads some Django templates and renders them, or uses the ORM to fetch some data`
import django
django.setup()

from book_list.models import BookList
```

---

In [14]:
import csv

check_limit = 2

with open('../data/books.csv', 'r', encoding='utf-8-sig') as cf:
    dataset = list(csv.DictReader(cf))

print(f'''총 데이터 수:{len(dataset)}
클래스: {type(dataset).__name__}
샘플 데이터 확인: {dataset[:check_limit]}''')

총 데이터 수:100000
클래스: list
샘플 데이터 확인: [{'title': 'SQL 자격검정 실전문제', 'primary_author': '한국데이터산업진흥원', 'publisher': '한국데이터산업진흥원', 'published_date': '2023년 12월', 'list_price': '18000', 'detail_url': 'https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=332583104', 'author_count': '1', 'authors': '한국데이터산업진흥원'}, {'title': '챗GPT·제미나이·클로드까지 모두를 위한 AI', 'primary_author': '지현이', 'publisher': '시프트', 'published_date': '2026년 7월', 'list_price': '22000', 'detail_url': 'https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=397038766', 'author_count': '1', 'authors': '지현이'}]


---

# 실행 시 발생 이슈

## 원인
Django는 async에서의 ORM 호출을 막습니다.
Jupyter Shell은 async로 취급되므로 이러한 동작을 수행할 수 없습니다.

## 해결방법
각 테스트.py를 생성하고 각각의 결과를 취합한다.

```text
---------------------------------------------------------------------------
SynchronousOnlyOperation                  Traceback (most recent call last)
Cell In[15], line 16
     12 # 데이터 적재
     13 for data in dataset:
     14     start_one = time.perf_counter()
     15 
---> 16     BookList.objects.create(
     17         title=data['title'],
     18         primary_author=data['primary_author'],
     19         publisher=data['publisher'],

File /opt/homebrew/Caskroom/miniforge/base/envs/bootcamp/lib/python3.13/site-packages/django/db/models/manager.py:87, in BaseManager._get_queryset_methods.<locals>.create_method.<locals>.manager_method(self, *args, **kwargs)
     85 @wraps(method)
     86 def manager_method(self, *args, **kwargs):
---> 87     return getattr(self.get_queryset(), name)(*args, **kwargs)

File /opt/homebrew/Caskroom/miniforge/base/envs/bootcamp/lib/python3.13/site-packages/django/db/models/query.py:713, in QuerySet.create(self, **kwargs)
    711 obj = self.model(**kwargs)
    712 self._for_write = True
--> 713 obj.save(force_insert=True, using=self.db)
    714 obj._state.fetch_mode = self._fetch_mode
    715 return obj

File /opt/homebrew/Caskroom/miniforge/base/envs/bootcamp/lib/python3.13/site-packages/django/db/models/base.py:904, in Model.save(self, force_insert, force_update, using, update_fields)
...
---> 24         raise SynchronousOnlyOperation(message)
     25 # Pass onward.
     26 return func(*args, **kwargs)

SynchronousOnlyOperation: You cannot call this from an async context - use a thread or sync_to_async.
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...
```

---

# 실험 1. 전체 데이터를 한건씩 삽입하기

아래 코드는 실제 테스트에 사용된 코드입니다.

```python

import time
from book_list.models import BookList

# time_list 초기화
time_list = []

# 시간 측정 시작
start_total = time.perf_counter()

# 데이터 적재
for data in dataset:
    start_one = time.perf_counter()

    BookList.objects.create(
        title=data['title'],
        primary_author=data['primary_author'],
        publisher=data['publisher'],
        published_date=data['published_date'],
        list_price=data['list_price'],
        detail_url=data['detail_url'],
        author_count=data['author_count'],
    )

    end_one = time.perf_counter()
    time_list.append(end_one - start_one)

# 시간 측정 종료
end_total = time.perf_counter()

print(f"회당 평균 소요시간: {sum(time_list) / len(time_list):.6f}초")
print(f"총 소요시간: {end_total - start_total:.6f}초")

```
---

# 실험 2. 전체 데이터를 한건씩 삽입하기

아래 코드는 실제 테스트에 사용된 코드입니다.

```python
import time
from book_list.models import BookList

book_list = []

# 시간 측정 시작
start_total = time.perf_counter()

# 인스턴스 생성

start_make_instance = time.perf_counter()

for data in dataset:

    book_list.append(
        BookList(
            title=data['title'],
            primary_author=data['primary_author'],
            publisher=data['publisher'],
            published_date=data['published_date'],
            list_price=data['list_price'],
            detail_url=data['detail_url'],
            author_count=data['author_count'],
        )
    )

end_make_instance = time.perf_counter()

# 전체 데이터 삽입
start_insert = time.perf_counter()

BookList.objects.bulk_create(book_list)

end_insert = time.perf_counter()

# 시간 측정 종료
end_total = time.perf_counter()

print(f"인스턴스 생성 소요시간: {end_make_instance - start_make_instance):.6f}초")
print(f"DB 삽입 소요시간: {end_insert_time - start_insert_time:.6f}초")
print(f"총 소요시간: {end_total - start_total:.6f}초")
```
---

# 실험 3. batch size를 조절하며 데이터 삽입하기

아래 코드는 실제 테스트에 사용된 코드입니다.

```python
import time
from book_list.models import BookList

book_list = []
batch_size = 10

# 시간 측정 시작
start_total = time.perf_counter()

# 인스턴스 생성

start_make_instance = time.perf_counter()

for data in dataset:

    book_list.append(
        BookList(
            title=data['title'],
            primary_author=data['primary_author'],
            publisher=data['publisher'],
            published_date=data['published_date'],
            list_price=data['list_price'],
            detail_url=data['detail_url'],
            author_count=data['author_count'],
        )
    )

end_make_instance = time.perf_counter()

# 전체 데이터 삽입
start_insert = time.perf_counter()

BookList.objects.bulk_create(book_list)

end_insert = time.perf_counter()

# 시간 측정 종료
end_total = time.perf_counter()

print(f"인스턴스 생성 소요시간: {end_make_instance - start_make_instance):.6f}초")
print(f"DB 삽입 소요시간: {end_insert_time - start_insert_time:.6f}초")
print(f"총 소요시간: {end_total - start_total:.6f}초")
```
---

# 전체 데이터 삭제하기
#### [Other QuerySet methods](https://docs.djangoproject.com/en/6.0/topics/db/queries/?utm_source=chatgpt.com#other-queryset-methods)
#### [Use QuerySet.update() and delete()](https://docs.djangoproject.com/en/6.0/topics/db/optimization/#use-foreign-key-values-directly)

아래 코드는 실제 테스트에 사용된 코드입니다.

```python

from book_list.models import BookList

BookList.objects.all().delete()
```
---

작성하고보니 각 실험을 10번씩 반복하고 평균을 내고 pandas 분석을 하는게 좋을 것 같은데...
하나의 파이프라인으로 묶어서 실행하려면 전부 모듈화하고 한번에 돌리는게 나을 것 같다는 생각이...?